# Week 13

Convolutional Neural Network and Residual Nets

In [ ]:
!wget -q https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/data_utils.py
!wget -q https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/image_utils.py
!wget -q https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/nn_utils.py

!wget -qO- https://github.com/PSAM-5020-2026S-A/5020-utils/releases/latest/download/lfw.tar.gz | tar xz

In [ ]:
from torch import nn, Tensor, float32 as t_float32, uint8 as t_uint8
from torch.optim import SGD

from torchvision.models import resnet34, ResNet34_Weights
from torchvision.transforms import v2

from data_utils import LFWUtils, classification_error, display_confusion_matrix
from image_utils import make_image
from nn_utils import get_labels, get_num_params, display_activation_grids, display_kernel_grids

## Spatial Information

Our fully-connected layers do ok for image datasets, but they are not very efficient.

There are $2$ main problems with using this approach to extract information about images:

### Every pixel is connected to every other pixel

Consider the first layer after the input layer: every neuron gets information about every pixel. This means that the content at the top-left corner of our image is connected to the content at the bottom-right corner, which is inefficient. We probably don't need our network to consider the entire content of the image at once in order to make decisions. It jumbles the pixel order and just makes the process harder. We might be better off telling our network to consider groups of neighboring pixels, since it's most likely for visual features to come from pixels that are near each other. In other words, we want to extract and preserve some kind of relative _Locality_ from our pixels.

### Not all Arnolds are the same

Let's say our network learned how to classify an Arnold Schwarzenegger that's closer to the left side of the image. If it wants to detect Arnolds on the right side of the image, or towards the top, it has to learn how to activate neurons that are associated with those sections of the image. This is also inefficient because it has to relearn to detect the same thing again, just because it's somewhere else in the image.

Again, what we would like to do is group neighboring pixels, and have the groups go through similar neurons, so that any kind of learning can be applied independent of where shapes are located in the image. The technical name for this property is _Translation Invariance_.

## Convolutions

We could try to come up with our own architecture and write some code for a neural network that doesn't fully connect our pixels, but rather considers neighboring regions of our image in groups of neurons.

But, luckily, some maths combined with intuition from old-school computer vision and feature engineering can help us here.

There's a type of mathematical operation called a convolution that combines $2$ arbitrary functions into a new function that basically has information about all the possible combinations of inputs for the $2$ original functions.

<!-- The math looks like this for the $1D$ continuous case:<br> -->
<!-- $\displaystyle (f * g)(\tau) = \int_{-\infty}^{\infty}{f(\tau)\ g(1 - \tau)\ d\tau}$ -->

The math looks like this for the $1D$ case:<br>
$\displaystyle (f * g)[n] = \sum_{k=-K}^{K}{f[k]\ \ g[n - k]}$


<br>And the $2D$ case:<br>
$\displaystyle (f * g)[n_x, n_y] = \sum_{k_x=-K}^{K}{\sum_{k_y=-K}^{K}{f[k_x, k_y]\ \ g[n_x - k_x, n_y - k_y]}}$

For the practical, intuitive, definition of this operation when dealing with images, $f[\ ]$ is an image tensor and $g[\ ]$ can be different, but specific, combinations of numbers organized into $2D$ matrices called kernels.

When we _convolve_ the image with the kernel, we calculate every possible overlap of our kernel with the image and, depending on the numbers we choose for the kernel, can extract different types of features from our pixels.

<!-- <img src="./imgs/kernel_slide.jpg" height="256px"/> -->
<img src="https://i.postimg.cc/wMzHtFnJ/kernel-slide.jpg" height="256px"/>

[SOME ANIMATIONS](https://hannibunny.github.io/mlbook/neuralnetworks/convolutionDemos.html)


Classic image processing kernels for sharpening an image and extracting edges:

<!-- <img src="./imgs/image_kernels.jpg" height="300px"/> -->
<img src="https://i.postimg.cc/fTKWdnzb/image-kernels.jpg" height="300px"/>

The nice thing about these kernels is that they operate on neighboring pixels by default, so they already take into account the _locality_ of the features they're trying to detect.

We can now set up a neural network that is a collection of $2D$ image kernels, and let our training algorithm learn parameters for these kernels based on the training data. We don't have to specify that we want an edge-detection kernel, or a curved-shape kernel, or a horizontal blur kernel... the network will learn the kernels that it needs.

And, since the same kernel slides over an entire image during convolution, once the network learns to extract lines on the left side of the image, it also knows how to extract lines on the right side of the image, or on top, or anywhere else. The parameters to the kernel are the same, they just get applied to different neighborhoods of pixels.

If we combine our bank of kernels with another operation to reduce the size of our image as it moves through the network, we can create a type of dynamic filtering that detects whether certain features are present on our image.

<!-- <img src="./imgs/cnn_layers.jpg" height="320px"/> -->
<img src="https://i.postimg.cc/rpdq7DSd/cnn-layers.jpg" height="320px"/>

Then, after we have reduced our feature maps to a small-enough shape we ca use fully-connected layers to finalize our classification.

<!-- <img src="./imgs/cnn_fc.jpg" height="320px"/> -->
<img src="https://i.postimg.cc/XYYjcHqk/cnn-fc.jpg" height="320px"/>

### New Network, New Shape

Right now our input samples are lists of pixels, organized in a `tensor` of shape $(1, 22100)$. In order to use the convolution layers we actually need the images in a $2D$ tensor of shape $(170, 130)$.

CNNs are even more particular about channels: we not only have to be explicit about the number of channels on our images, but this info also has to be in the first dimension of our image tensors.

So, instead of working with the original image shape where each pixel keeps the values for its channels together, like:<br>
$H \times W \times C$ (`height` by `width` by `channel`)

we now have to use a `tensor` made up of single-channel images: all the RED values, then all the GREEN values, then all the BLUE values... so a $3D$ `tensor` of shape: $C \times H \times W$. For our `LFW` images this is $(1, 170, 130)$.

The [`permute()`](https://pytorch.org/docs/stable/generated/torch.permute.html) or [`movedim()`](https://pytorch.org/docs/stable/generated/torch.movedim.html#torch.movedim) functions can be used to reorganize the order of a `tensor`'s dimensions.

In the end, our dataset should be in a `tensor` of shape:<br>
$N \times C \times H \times W$ (`samples`, `channels`, `height`, `width`).

In [ ]:
train, test = LFWUtils.train_test_split(dir="./data/image/lfw/cropped", test_pct=0.5)

In [ ]:
len(train["pixels"])

In [ ]:
x_train = Tensor(train["pixels"])
x_train.shape

x_train = x_train.reshape(445, 170, 130, 1)

x_train.shape

x_train = x_train.permute(0, 3, 1, 2)

x_train.shape

We're working with grayscale images in this exercise, but the code below should also work for `RGB` images by changing `n_channels` to $3$.

In [ ]:
LFWUtils.IMAGE_SIZE

In [ ]:
iw,ih = LFWUtils.IMAGE_SIZE
n_channels = 1
n_samples = len(train["pixels"])

x_train = Tensor(train["pixels"]) # 445 x 22100
x_train = x_train.reshape(n_samples, ih, iw, n_channels) # 445 x 170 x 130 x 1
x_train = x_train.permute(0,3,1,2) # 445 x 1 x 170 x 130

# reshape 438 x 22100 into 438 x 1 x 170 x 130
x_test = Tensor(test["pixels"]).reshape(-1, ih, iw, n_channels).movedim(-1, 1)

y_train = Tensor(train["labels"]).long()
y_test = Tensor(test["labels"]).long()

In [ ]:
x_train.shape, x_test.shape

In [ ]:
print("Dataset Samples")
print("\tTrain:", len(x_train))
print("\tTest:", len(x_test))

print("\nDataset Shape:")
print(f"\tTrain:", list(x_train.shape))
print(f"\tTtest:", list(x_test.shape))

print("\nSample Shape:")
print(f"\tTrain:", list(x_train[0].shape))
print(f"\tTest:", list(x_test[0].shape))

### Check Image

We have to undo the channel permutation and change $C \times H \times W$ (`channels`, `height`, `width`) back into $H \times W \times C$ (`height`, `width`, `channels`).

Since these are grayscale images we could also just reshape them into $H \times W$:

`img_t = x_train[idx].reshape(-1, x_train.shape[1])`

This only works for grayscale images though.

In [ ]:
mimg = x_train[10][0]

mimg.shape
display(make_image(mimg, width=x_train.shape[3]))

In [ ]:
idx = 0
img_t = x_train[idx].permute(1,2,0).reshape(-1, x_train.shape[1])
display(make_image(img_t, width=x_train.shape[3]))

### Define CNN Model

This is how we define a convolution layer:

`nn.Conv2d(Cin, Cout, kernel_size)`

Where `Cin` is the number of input channels, `Cout` is output channels, and `kernel_size` the width of our kernel.

We should still normalize the computations by batch, but this time using the $2D$ version of `BatchNorm()`, and after activation we perform the `MaxPool` operation, which takes the largest value in a $2 \times 2$ region of our activations and condenses them into denser representation of features.

In [ ]:
linear_length = 83664

model = nn.Sequential(
  nn.Conv2d(1, 16, 5),

  nn.Sigmoid(),

  nn.MaxPool2d(2),

  nn.Flatten(),

  nn.Linear(linear_length, len(LFWUtils.LABELS)),
)

optim = SGD(model.parameters(), lr=1e-4, momentum=0.9)

loss_fn = nn.CrossEntropyLoss()

out = model(x_train)

print("Input shape:", x_train.shape)
print("Output shape:", out.shape)
print("Parameters:", get_num_params(model))

In [ ]:
out[0]

### Put model and data on GPU (if available)

In [ ]:
model = model.to("cuda")

x_train = x_train.to("cuda")
x_test = x_test.to("cuda")

y_train = y_train.to("cuda")
y_test = y_test.to("cuda")

### Train and Evaluate

In [ ]:
for e in range(32):
  optim.zero_grad()
  labels_pred = model(x_train)
  loss = loss_fn(labels_pred, y_train)
  loss.backward()
  optim.step()

  if e % 4 == 3:
    train_predictions = get_labels(model, x_train)
    test_predictions = get_labels(model, x_test)
    train_error = classification_error(y_train.cpu(), train_predictions)
    test_error = classification_error(y_test.cpu(), test_predictions)
    print(f"Epoch: {e} loss: {loss.item():.4f}, train error: {train_error:.4f}, test error: {test_error:.4f}")

In [ ]:
train_predictions = get_labels(model, x_train)
test_predictions = get_labels(model, x_test)

print("train error", f"{classification_error(y_train.cpu(), train_predictions):.4f}")
print("test error", f"{classification_error(y_test.cpu(), test_predictions):.4f}")

display_confusion_matrix(y_train.cpu(), train_predictions, display_labels=LFWUtils.LABELS)
display_confusion_matrix(y_test.cpu(), test_predictions, display_labels=LFWUtils.LABELS)

In [ ]:
# TODO: experiment with the network above:
#       add layers, change the kernel size, number of channels, stride, activation function
#       add dropout, add batchnorm, change the pooling layer, etc...

linear_length = 72384

model = nn.Sequential(
  nn.Conv2d(1, 32, 5),

  nn.ReLU(),

  # nn.LayerNorm(1),

  nn.MaxPool2d(2),

  nn.Conv2d(32, 64, 5),

  nn.ReLU(),

  nn.MaxPool2d(2),

  nn.Flatten(),

  nn.Linear(linear_length, len(LFWUtils.LABELS)),
)

optim = SGD(model.parameters(), lr=1e-3, momentum=0.9)

loss_fn = nn.CrossEntropyLoss()

model = model.to("cuda")

out = model(x_train)

print("Input shape:", x_train.shape)
print("Output shape:", out.shape)
print("Parameters:", get_num_params(model))

In [ ]:
for e in range(32):
  optim.zero_grad()
  labels_pred = model(x_train)
  loss = loss_fn(labels_pred, y_train)
  loss.backward()
  optim.step()

  if e % 4 == 3:
    train_predictions = get_labels(model, x_train)
    test_predictions = get_labels(model, x_test)
    train_error = classification_error(y_train.cpu(), train_predictions)
    test_error = classification_error(y_test.cpu(), test_predictions)
    print(f"Epoch: {e} loss: {loss.item():.4f}, train error: {train_error:.4f}, test error: {test_error:.4f}")

In [ ]:
train_predictions = get_labels(model, x_train)
test_predictions = get_labels(model, x_test)

print("train error", f"{classification_error(y_train.cpu(), train_predictions):.4f}")
print("test error", f"{classification_error(y_test.cpu(), test_predictions):.4f}")

display_confusion_matrix(y_train.cpu(), train_predictions, display_labels=LFWUtils.LABELS)
display_confusion_matrix(y_test.cpu(), test_predictions, display_labels=LFWUtils.LABELS)

## Transfer Learning

The CNN architecture is so stable that models can be made to be very deep, some with $100\text{s}$ of layers.

The internal layers of these models are so abstract and generic that once a model has been trained on millions of data samples (images), it learns and retains information not only about the images on the dataset, but any visual pattern that it learned in the process.

It's not uncommon to use a previously trained model for a similar-but-different project, even if the images have nothing in common. Generic information about images can be transferred to new datasets and problem spaces.

<!-- <img src="./imgs/resnet_activation_00.jpg" height="300px" /> -->
<img src="https://i.postimg.cc/tR3twzmz/resnet-activation-00.jpg" height="300px" />

<!-- <img src="./imgs/resnet_activation_01.jpg" height="300px" /> -->
<img src="https://i.postimg.cc/hPKbB7kR/resnet-activation-01.jpg" height="300px" />

### Residual Networks

There are a couple of families of CNN networks that get used as the starting point for many different types of visual models (and also audio and text). One such architecture is [ResNet](https://arxiv.org/abs/1512.03385).

ResNet comes in a few sizes/depths, and PyTorch has at least [5 pre-trained ResNet models](https://pytorch.org/hub/pytorch_vision_resnet/) that we can use.

These PyTorch ResNet models were trained on the [ImageNet](https://image-net.org/download.php) dataset. This dataset has $1\text{,}281\text{,}167$ training images and classifies objects into $1\text{,}000$ classes.

We'll use the `ReNet34` model, which is not the largest, but will fit nicely into small GPUs.

<!-- <img src="./imgs/resnet34_00.jpg" width="900px" /> -->
<img src="https://i.postimg.cc/XNc8xdqy/resnet34-00.jpg" width="900px" />

<!-- <img src="./imgs/resnet34_01.jpg" width="900px" /> -->
<img src="https://i.postimg.cc/hP20Rn9D/resnet34-01.jpg" width="900px" />


### Instantiating ResNet

Is easy:

In [ ]:
model = resnet34(weights=ResNet34_Weights.DEFAULT)
print(get_num_params(model))
display(model)

### Adjust inputs

From https://pytorch.org/hub/pytorch_vision_resnet/:

_All pre-trained models expect input images normalized in the same way, i.e. mini-batches of 3-channel RGB images of shape (3 x H x W), where H and W are expected to be at least 224. The images have to be loaded in to a range of [0, 1] and then normalized using mean = [0.485, 0.456, 0.406] and std = [0.229, 0.224, 0.225]._

We can use `PyTorch` transformation functions to achieve this.

In [ ]:
res_transforms = v2.Compose([
  v2.ToDtype(t_uint8),
  v2.Resize(224),
  v2.Grayscale(3),
  v2.ToDtype(t_float32, scale=True),
  v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
x_train_res = res_transforms(x_train).to("cuda")
x_test_res = res_transforms(x_test).to("cuda")

print("Training dataset shape:", x_train_res.shape)
print("Image shape:", x_train_res[0].shape)

In [ ]:
x_train_res[0].min(), x_train_res[0].max()

### Modify output layer

The last layer of the default `ResNet34` model has $1\text{,}000$ outputs, one for each of the classes in the ImageNet dataset.

We're not using that dataset. We're classifying our images into one of $25$ classes, so we have to swap the last layer for one with $25$ outputs.

In [ ]:
model.fc = nn.Linear(model.fc.in_features, len(LFWUtils.LABELS))
model = model.to("cuda")

optim = SGD(model.parameters(), lr=5e-3, momentum=0.9)

loss_fn = nn.CrossEntropyLoss()

out = model(x_train_res[::3])

print("Input shape:", x_train_res.shape)
print("Output shape:", out.shape)
print("Parameters:", get_num_params(model))

In [ ]:
y_train = y_train.to("cuda")
y_test = y_test.to("cuda")

In [ ]:
batch_step = 3

for e in range(16):
  model.train()
  for batch_start in range(batch_step):
    optim.zero_grad()
    labels_pred = model(x_train_res[batch_start::batch_step])
    loss = loss_fn(labels_pred, y_train[batch_start::batch_step])
    loss.backward()
    optim.step()

  if e % 4 == 3:
    train_predictions = get_labels(model, x_train_res)
    test_predictions = get_labels(model, x_test_res)
    train_error = classification_error(y_train.cpu(), train_predictions)
    test_error = classification_error(y_test.cpu(), test_predictions)
    print(f"Epoch: {e} loss: {loss.item():.4f}, train error: {train_error:.4f}, test error: {test_error:.4f}")

In [ ]:
train_predictions = get_labels(model, x_train_res)
test_predictions = get_labels(model, x_test_res)
train_error = classification_error(y_train.cpu(), train_predictions)
test_error = classification_error(y_test.cpu(), test_predictions)
print(f"train error: {train_error:.4f}, test error: {test_error:.4f}")

display_confusion_matrix(y_train.cpu(), train_predictions, display_labels=LFWUtils.LABELS)
display_confusion_matrix(y_test.cpu(), test_predictions, display_labels=LFWUtils.LABELS)

### Visualize Layers

That worked really well. The information learned by the `ResNet` network on 1 million images seems to transfer to our classification of faces and we can leverage its pattern-recognition layers to build a more accurate model in a short amount of time.

Let's take a look at some of the filtered images in our `ResNet` model. We can do this with an untrained model, but it's better to look at one that has been recently trained.

When we displayed the model layers above we saw that the model has $4$ main groups of convolution layers. The further down the model we go, the smaller the images are, and the more abstract the activation patterns will be.

At the very last layer we might have $512$ _images_ that are only $4 \times 4$ pixels, but light-up under very specific conditions, like: is there a bird in the image ? is there a face ?

To see slightly larger images we'll look at some activations on layers $1$ and $2$.

We'll use the `hook` mechanism from `PyTorch` to add some auxiliary logic to the layers we are interested in looking at. This allows us to run some extra code on the layers inputs and outputs every time it processes an image.

Our `hook` function will just save the layers input and output tensors to external dictionaries that we cn visualize later.

In [ ]:
activations_in = {}
activations_out = {}
layer_kernels = {}

def get_activation(name):
  def hook(model, input, output):
    if name not in layer_kernels:
      layer_kernels[name] = model.weight.detach()
    activations_in[name] = input[0].detach()
    activations_out[name] = output.detach()
  return hook

model.conv1.register_forward_hook(get_activation("conv1"))
model.layer1[1].conv2.register_forward_hook(get_activation("layer1.1.conv2"))
model.layer1[2].conv2.register_forward_hook(get_activation("layer1.2.conv2"))
model.layer2[0].conv2.register_forward_hook(get_activation("layer2.0.conv2"))
model.layer4[0].conv2.register_forward_hook(get_activation("layer4.0.conv2"))
model.layer4[2].conv2.register_forward_hook(get_activation("layer4.2.conv2"))
model = model.to("cuda")

Once we have our `hook` in place we have to pass some data through the network so it saves the inputs and outputs for us.

In [ ]:
from torch import no_grad

with no_grad():
  model(x_train_res[0:128].to("cuda"))

Now we can display activations for specific images in the processed batch:

In [ ]:
img_idx = 0
channel_idx = 0

img_t = x_train[img_idx, channel_idx]

display(make_image(img_t, width=img_t.shape[-1]))
display_activation_grids(activations_out, img_idx)

And we can also visualize the kernel values in each layer.

These are tiny $3 \times 3$ or $7 \times 7$ convolution kernels that get multiplied to filter the images.

Other than the first one, where the kernels actually acted on 3-channel layers, the colors on the other kernels is artificial. They're the result of combining the kernels into groups of $3$, but in reality they are 64-channel kernels.

In [ ]:
display_kernel_grids(layer_kernels)